In [ ]:
import importlib, sys

def reimport_trackers():
    """Force-reload all trackers submodules already in sys.modules."""
    mods = [k for k in sys.modules if k.startswith("trackers")]
    for m in sorted(mods, reverse=True):
        importlib.reload(sys.modules[m])
    print(f"Reloaded {len(mods)} trackers module(s).")

# Re-ID Evaluation — OSNet on Market-1501 and MSMT17

This notebook reproduces the standard **same-domain** re-ID benchmark numbers for OSNet x1.0.
Each dataset is evaluated with the checkpoint that was *trained on that same dataset*
(from the [torchreid model zoo](https://kaiyangzhou.github.io/deep-person-reid/MODEL_ZOO)),
which is the only fair comparison against published numbers.

1. **Market-1501** (~1.3 GB, ~10 min on T4) — Expected: ~94.2 Rank-1 / 82.6 mAP.
2. **MSMT17** (~4 GB, ~40 min on T4) — Expected: ~74.9 Rank-1 / 43.8 mAP.

> **Note:** the default `ReIDModel.from_pretrained()` checkpoint is OSNet trained on
> MSMT17 with `combineall=True` (train+test combined). Evaluating *that* checkpoint on
> MSMT17 leaks the test set (near-100% scores) and on Market-1501 measures cross-domain
> transfer (~61 R1). To reproduce paper numbers we instead load each dataset's own
> same-domain checkpoint below.

> **Runtime:** `Runtime → Change runtime type → T4 GPU` before running.

---

## 1. Install

In [ ]:
# Install the trackers library from the feature branch + reid optional deps.
# The [reid] extra pulls in torch, torchvision, timm, and huggingface-hub.
!pip install -q --upgrade pip
!pip install -q --no-cache-dir 'trackers[reid] @ git+https://github.com/roboflow/trackers.git@feat/reid-phase1'

reimport_trackers()

In [ ]:
import warnings
import numpy as np

# Confirm GPU is available.
import torch
print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}  |  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

## 2. Setup

Imports the re-ID API and defines a small helper to fetch the per-dataset OSNet x1.0
checkpoints from the torchreid model zoo (hosted on Google Drive, ~9 MB each).
The matching checkpoint is loaded inside each dataset's section below via
`ReIDModel.from_checkpoint(...)`.

In [ ]:
!pip install -q gdown
import os
import gdown

from trackers.core.reid import ReIDModel, ReidEvaluator

CKPT_DIR = "/content/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def download_checkpoint(gdrive_id: str, filename: str) -> str:
    """Download an OSNet checkpoint from the torchreid model zoo (Google Drive)."""
    path = os.path.join(CKPT_DIR, filename)
    if not os.path.exists(path):
        gdown.download(id=gdrive_id, output=path, quiet=False)
    return path

print("Setup complete.")

---
## 3. Market-1501

**Expected numbers (OSNet x1.0 trained on Market-1501, torchreid model zoo):** Rank-1 ≈ 94.2 %  |  mAP ≈ 82.6 %

Market-1501 is ~1.3 GB. We download it via gdown (official Google Drive mirror).
If gdown fails, see the alternative download cell below.

In [ ]:
!pip install -q gdown
import gdown, zipfile, os

MARKET_ZIP = "/content/Market-1501.zip"
MARKET_DIR = "/content/Market-1501-v15.09.15"

if not os.path.exists(MARKET_DIR):
    # Google Drive file ID for Market-1501-v15.09.15.zip
    gdown.download(id="0B8-rUzbwVRk0c054eEozWG9COHM", output=MARKET_ZIP, quiet=False)
    with zipfile.ZipFile(MARKET_ZIP, "r") as zf:
        zf.extractall("/content")
    print("Extracted to", MARKET_DIR)
else:
    print("Already downloaded.")

In [ ]:
# Alternative download if gdown fails (paste the unzip path that matches your mirror):
# !wget -q -O /content/Market-1501.zip "<YOUR_MIRROR_URL>"
# !unzip -q /content/Market-1501.zip -d /content

In [ ]:
from trackers.core.reid import load_market1501

query_m, gallery_m = load_market1501(MARKET_DIR)
print(f"Market-1501 — query: {len(query_m):,}  |  gallery: {len(gallery_m):,}")

In [ ]:
# Load the OSNet x1.0 checkpoint trained on Market-1501 (torchreid model zoo).
market_ckpt = download_checkpoint("1vduhq5DpN2q1g4fYEZfPI17MJeh9qyrA", "osnet_x1_0_market1501.pth")
model_market = ReIDModel.from_checkpoint(market_ckpt)
evaluator_market = ReidEvaluator(model_market, batch_size=256)

result_market = evaluator_market.evaluate(query_m, gallery_m)

print("\nMarket-1501 results:")
print(f"  mAP    : {result_market.metrics.map:.1f}%   (model zoo: ~82.6%)")
print(f"  Rank-1 : {result_market.metrics.rank1:.1f}%   (model zoo: ~94.2%)")
print(f"  Rank-5 : {result_market.metrics.rank5:.1f}%")
print(f"  Rank-10: {result_market.metrics.rank10:.1f}%")
print(f"  mINP   : {result_market.metrics.minp:.1f}%")

---
## 4. MSMT17

**Expected numbers (OSNet x1.0 trained on MSMT17, torchreid model zoo):** Rank-1 ≈ 74.9 %  |  mAP ≈ 43.8 %

### Option A — download directly in Colab (recommended)

Downloads `MSMT17_V1.zip` (~2.56 GB) from the community Hugging Face mirror
[`xianpeijie/MSMT17_V1`](https://huggingface.co/datasets/xianpeijie/MSMT17_V1)
using `huggingface_hub` (already installed as part of the `[reid]` extra).

In [ ]:
# Option A: download from Hugging Face community mirror (~2.56 GB)
from huggingface_hub import hf_hub_download
import zipfile, os

MSMT17_DIR = "/content/MSMT17_V1"

if not os.path.exists(MSMT17_DIR):
    msmt17_zip = hf_hub_download(
        repo_id="xianpeijie/MSMT17_V1",
        filename="MSMT17_V1.zip",
        repo_type="dataset",
        local_dir="/content",
    )
    print("Extracting MSMT17 (~2.56 GB, may take a few minutes)…")
    with zipfile.ZipFile(msmt17_zip, "r") as zf:
        zf.extractall("/content")
    print("Done →", MSMT17_DIR)
else:
    print("Already extracted.")

In [ ]:
from trackers.core.reid import load_msmt17

query_ms, gallery_ms = load_msmt17(MSMT17_DIR)
print(f"MSMT17 — query: {len(query_ms):,}  |  gallery: {len(gallery_ms):,}")

In [ ]:
# Load the OSNet x1.0 checkpoint trained on MSMT17 (standard split, torchreid model zoo).
msmt17_ckpt = download_checkpoint("112EMUfBPYeYg70w-syK6V6Mx8-Qb9Q1M", "osnet_x1_0_msmt17.pth")
model_msmt17 = ReIDModel.from_checkpoint(msmt17_ckpt)
evaluator_msmt17 = ReidEvaluator(model_msmt17, batch_size=256)

result_msmt17 = evaluator_msmt17.evaluate(query_ms, gallery_ms)

print("\nMSMT17 results:")
print(f"  mAP    : {result_msmt17.metrics.map:.1f}%   (model zoo: ~43.8%)")
print(f"  Rank-1 : {result_msmt17.metrics.rank1:.1f}%   (model zoo: ~74.9%)")
print(f"  Rank-5 : {result_msmt17.metrics.rank5:.1f}%")
print(f"  Rank-10: {result_msmt17.metrics.rank10:.1f}%")
print(f"  mINP   : {result_msmt17.metrics.minp:.1f}%")

---
## 5. Summary table

In [ ]:
print(f"{'Dataset':<14} {'mAP':>8} {'Rank-1':>8} {'Rank-5':>8} {'Rank-10':>9} {'mINP':>8}")
print("-" * 55)

def _row(name, m):
    return f"{name:<14} {m.map:>7.1f}% {m.rank1:>7.1f}% {m.rank5:>7.1f}% {m.rank10:>8.1f}% {m.minp:>7.1f}%"

print(_row("Market-1501", result_market.metrics))
print(_row("MSMT17",      result_msmt17.metrics))
print("-" * 55)
print("Model-zoo targets: Market-1501 R1≈94.2 mAP≈82.6  |  MSMT17 R1≈74.9 mAP≈43.8")